In [26]:
%matplotlib widget

import os
import numpy as np
from PIL import Image
import cv2
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patheffects as pe
import pytesseract
import matplotlib.patches as mpatches
from matplotlib.patches import Rectangle, Polygon
import image_utils
from get_cards_tablet import card_peek_width, card_width, get_cards, card_height
import cv2
import numpy as np
import os
import matplotlib.pyplot as plt
from image_utils import load_rdb, save_rgb_png
from android_capture import ScreenCapture
import time
import image_utils
import ocr_cards
import matplotlib.pyplot as plt
from android_capture import ScreenCapture
import time
import image_utils
import get_cards_tablet
import ocr_cards
import matplotlib.pyplot as plt
from android_actions import AndroidBJTabletActor
from blackjack.actions import PlayerAction
from blackjack.cards import Card, Rank
from blackjack.hand import Hand

In [27]:
class Counter:
    def __init__(self, n_decks, current_penetration=0):
        self.n_decks = n_decks
        self.remaining_cards = n_decks * 52
        self.remaining_cards -= int(round(self.remaining_cards * current_penetration))
        self.running_count = 0
    
    def reset(self):
        self.remaining_cards = self.n_decks * 52
        self.running_count = 0

    def update_count(self, rank_value):
        # Tag values: 2–6 = +1, 7–9 = 0, 10–A = −1
        if rank_value < 7:
            self.running_count += 1
        elif rank_value > 9:
            self.running_count -= 1
        self.remaining_cards -= 1
    
    def get_true_count(self):
        remaining_decks = self.remaining_cards / 52
        if remaining_decks <= 0:
            return None
        return self.running_count / remaining_decks
    
    def get_integer_tc(self):
        # use integer TCs rounded toward zero
        # 1.9 -> 1 | -1.8 -> -1
        true_count = self.get_true_count()
        if true_count is None:
            return None
        return int(true_count)

In [28]:
class Strategy:
    def __init__(self, hard_table, soft_table, split_table, tc_hard_changes, tc_soft_changes):
        pass

    def get_actions(self, player_value, is_soft, is_pair, dealer_card, true_count=0):
        # Only if you don’t split do you then apply the total-based indices to the resulting hand.

        if is_soft:
            table = self.soft_table
            tc_changes = self.tc_soft_changes
        else:
            table = self.hard_table
            tc_changes = self.tc_hard_changes

        actions = None
        if player_value in tc_changes:
            if dealer_card in tc_changes[player_value]:
                tc_min, tc_max, _actions = tc_changes[player_value][dealer_card]
                if tc_min <= true_count <= tc_max:
                    actions = _actions

        if actions is None:
            actions = table[player_value][dealer_card]
            
        if true_count >= self.insurance_count:
            actions = [PlayerAction.TAKE_INSURANCE] + actions        
        
        return actions

In [ ]:

class Player:
    def __init__(self):
        self.actor = AndroidBJTabletActor()
        self.scr_taker = ScreenCapture(save_folder=None)  # "/media/maxim/T7/frames/")
        self.dealer_hand = Hand()
        self.player_hands = [Hand()]
        self.round = BJRound()
        self.active_hand = get_cards_tablet.ActiveHand.NONE

        game_img = self.scr_taker.get_screen()
        initial_penetration = get_cards_tablet.get_shoe_penetration(game_img)
        print("initial_penetration ", initial_penetration)
        self.card_counter = Counter(n_decks=6, current_penetration=initial_penetration) 

    def is_pre_shuffle(self):
        game_img = self.scr_taker.get_screen()
        return get_cards_tablet.is_pre_shuffle(game_img)

    def reset_hands(self):
        self.dealer_hand = Hand()
        self.player_hands = [Hand()]
        self.active_hand = get_cards_tablet.ActiveHand.NONE

    def play_round(self, bet=1000):
        self.reset_hands()

        game_img = self.scr_taker.get_screen()
        while not get_cards_tablet.is_table_empty(game_img):
            time.sleep(0.5)
            game_img = self.scr_taker.get_screen()
        self.actor.place_bet(bet)
        self.actor.deal()
        self.wait_for_initial_cards()

        print("initial hand")
        print(self.dealer_hand)
        print(self.player_hands)

        print("true count: ", self.card_counter.get_true_count())

        player_had_bj = self.player_hands[0].is_natural_blackjack()

        if not player_had_bj:
            if self.dealer_hand[0].rank_value() == 10:
                self.wait_for_dealer_bj_or_action_request()
            elif self.dealer_hand[0].rank_value() == 11:
                # if dealer has A, insurance is always offered
                self.wait_for_insurance_option()
                self.actor.refuse_insurance()
                self.wait_for_dealer_bj_or_action_request()
            else:
                self.wait_for_action_request()
        else:
            if self.dealer_hand[0].rank_value() == 11:
                self.wait_for_insurance_option()
                self.actor.refuse_insurance()
            # sometimes dealer both 2 cards are captured by wait_for_initial_cards
            # instead of one
            self.wait_for_dealer_full_hand(only_two_cards=True)

        dealer_has_bj = self.dealer_hand.is_natural_blackjack()
        if player_had_bj or dealer_has_bj:
            self.end_round()
            return
        
        if self.player_hands[0].get_best_value() < 17:
            self.hit()
            self.wait_for_hit_card()

        player_bust = self.player_hands[0].is_bust()
        player_21 = self.player_hands[0].get_best_value() == 21
        if not player_bust and not player_21:
            self.stand()
            
        # when player is bust dealer shows only one extra card
        self.wait_for_dealer_full_hand(only_two_cards=player_bust)
        self.end_round()

    def double(self):
        self.wait_for_action_btn()
        self.actor.double()

    def split(self):
        self.wait_for_action_btn()
        self.actor.split()

    def hit(self):
        self.wait_for_action_btn()
        self.actor.hit()

    def stand(self):
        self.wait_for_action_btn()
        self.actor.stand()

    def wait_for_action_btn(self):
        while True:
            game_img = self.scr_taker.get_screen()
            can_stand = get_cards_tablet.can_hit_stand(game_img)
            if can_stand:
                break

    def end_round(self):
        print("final hand")
        print(self.dealer_hand)
        print(self.player_hands)

        self.actor.rebuy()

        for c in self.dealer_hand.cards:
            self.card_counter.update_count(c.rank_value())
        for h in self.player_hands:
            for c in h.cards:
                self.card_counter.update_count(c.rank_value())
        

    def wait_for_initial_cards(self):
        print("wait_for_initial_cards")
        game_img = self.scr_taker.get_screen()
        dealer_cards_new = get_cards_tablet.get_dealer_cards(game_img)
        middle_cards_new = get_cards_tablet.get_player_cards_middle(game_img)

        dealer_cards_set = set(["".join(dealer_cards_new)])
        middle_cards_set = set(["".join(middle_cards_new)])

        cards_confirmed = False
        while True:
            if (
                cards_confirmed 
                and len(dealer_cards_new) >= 1
                and len(middle_cards_new) >= 2
            ):
                break

            game_img = self.scr_taker.get_screen()
            dealer_cards_new = get_cards_tablet.get_dealer_cards(game_img)
            middle_cards_new = get_cards_tablet.get_player_cards_middle(game_img)

            dealer_cards_str = "".join(dealer_cards_new)
            middle_cards_str = "".join(middle_cards_new)

            if dealer_cards_str in dealer_cards_set and middle_cards_str in middle_cards_set:
                cards_confirmed = True
            else:
                cards_confirmed = False
                dealer_cards_set.add(dealer_cards_str)
                middle_cards_set.add(middle_cards_str)

        self.dealer_hand = Hand([Card.from_str(s) for s in dealer_cards_new])
        self.player_hands = [Hand([Card.from_str(s) for s in middle_cards_new])]


    def wait_for_insurance_option(self):
        print("wait_for_insurance_option")
        game_img = self.scr_taker.get_screen()
        while not get_cards_tablet.is_insurance_offered(game_img):
            game_img = self.scr_taker.get_screen()


    def wait_for_action_request(self):
        print("wait_for_action_request")
        while True:
            if self.active_hand != get_cards_tablet.ActiveHand.NONE:
                break
            game_img = self.scr_taker.get_screen()
            self.active_hand = get_cards_tablet.get_active_hand(game_img)


    def wait_for_dealer_bj_or_action_request(self):
        # after insurance refused dealer either shows bj or points to my hand
        print("wait_for_dealer_bj_or_action_request")
        game_img = self.scr_taker.get_screen()
        dealer_cards_new = get_cards_tablet.get_dealer_cards(game_img)
        dealer_cards_set = set(["".join(dealer_cards_new)])
        
        dealer_confirmed = False
        while True:
            # dealer shows blackjack
            if dealer_confirmed and len(dealer_cards_new) >= 2:
                break
            # dealer asks for action
            if self.active_hand != get_cards_tablet.ActiveHand.NONE:
                break
                
            game_img = self.scr_taker.get_screen()
            self.active_hand = get_cards_tablet.get_active_hand(game_img)
            dealer_cards_new = get_cards_tablet.get_dealer_cards(game_img)
            
            dealer_cards_str = "".join(dealer_cards_new)
            if dealer_cards_str in dealer_cards_set:
                dealer_confirmed = True
            else:
                dealer_cards_set.add(dealer_cards_str)
                dealer_confirmed = False
        
        if len(dealer_cards_new) >= 2: 
            self.dealer_hand = Hand([Card.from_str(s) for s in dealer_cards_new])
        

    
    def wait_for_dealer_full_hand(self, only_two_cards=False):
        print("wait_for_dealer_full_hand")
        if only_two_cards and len(self.dealer_hand) >= 2:
            return
        
        game_img = self.scr_taker.get_screen()

        dealer_cards_new = get_cards_tablet.get_dealer_cards(game_img)
        dealer_cards_set = set(["".join(dealer_cards_new)])
        current_hand = Hand([Card.from_str(s) for s in dealer_cards_new])
        
        dealer_confirmed = False
        while True:
            if dealer_confirmed: 
                if only_two_cards:
                    if len(current_hand) >= 2:
                        break
                else:
                    if current_hand.is_bust() or current_hand.get_best_value() >= 17:
                        break
        
            game_img = self.scr_taker.get_screen()
            dealer_cards_new = get_cards_tablet.get_dealer_cards(game_img)
            dealer_cards_str = "".join(dealer_cards_new)
            
            if dealer_cards_str in dealer_cards_set:
                dealer_confirmed = True
                current_hand = Hand([Card.from_str(s) for s in dealer_cards_new])
            else:
                dealer_confirmed = False
                dealer_cards_set.add(dealer_cards_str)
                current_hand = None

            print("dealer_cards_new", dealer_cards_new, current_hand)
        self.dealer_hand = current_hand


    def wait_for_hit_card(self):
        print("wait_for_hit_card")
        hand_idx = 1 if self.active_hand == get_cards_tablet.ActiveHand.LEFT else 0
        hand = self.player_hands[hand_idx]
        
        # 4 cards in each line 
        # assume hand is not empty, length >= 2
        n_cards_last_line = len(hand) % 4 + 1
        # 6th card covers cards 1,2,3,4 (entire previous line)
        visible_tail_idx = slice(-n_cards_last_line, None)
        if n_cards_last_line == 4:
            # 5th card covers cards 1,2,3
            visible_tail_idx = slice(-1, None)
        visible_old_tail = [str(c) for c in hand[visible_tail_idx]]
        print("visible_old_tail", visible_old_tail)
        game_img = self.scr_taker.get_screen()
        hand_cards_new = get_cards_tablet.get_player_cards(game_img, self.active_hand)
        hand_cards_set = set(["".join(hand_cards_new)])
        
        hand_confirmed = False
        while True:
            if (
                hand_confirmed 
                and hand_cards_new[:-1] == visible_old_tail
                and len(hand_cards_new) == len(visible_old_tail) + 1
            ):
                break

            game_img = self.scr_taker.get_screen()
            hand_cards_new = get_cards_tablet.get_player_cards(game_img, self.active_hand)
            print("hand_cards_new", hand_cards_new)
            
            hand_cards_str = "".join(hand_cards_new)
            if hand_cards_str in hand_cards_set:
                hand_confirmed = True
            else:
                hand_confirmed = False
                hand_cards_set.add(hand_cards_str)
        
        self.player_hands[hand_idx].add_card(Card.from_str(hand_cards_new[-1]))


In [30]:
all_cards = []

In [31]:
p = Player()
while not p.is_pre_shuffle():
    print("new hand")
    p.play_round()
    all_cards.extend(p.player_hands[0].cards)
    all_cards.extend(p.dealer_hand.cards)
    print("hand ended")
    time.sleep(1)


initial_penetration  1.0
new hand
wait_for_initial_cards
initial hand
[Kc] (10)
[<Hand: [7c2h] (9)>]
true count:  None
wait_for_dealer_bj_or_action_request
wait_for_hit_card
visible_old_tail ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_new ['7c', '2h']
hand_cards_n

In [45]:
p = Player()
game_img = p.scr_taker.get_screen()
initial_penetration = get_cards_tablet.get_shoe_penetration(game_img)

initial_penetration  0.030000000000000027


In [33]:
p.scr_taker.close()

In [34]:
cards = list(all_cards)

In [46]:
from collections import Counter

In [47]:
card_count = Counter()
for card in cards:
    card_count[str(card)] += 1

In [48]:
card_count

Counter({'7c': 6,
         '3s': 6,
         'Kc': 6,
         '7d': 6,
         '6d': 6,
         'Jd': 6,
         'Kh': 6,
         '9d': 6,
         'Td': 6,
         '9h': 6,
         'Qc': 6,
         'Qd': 6,
         'Ad': 6,
         'Ts': 6,
         '8c': 6,
         'Qh': 6,
         '4d': 5,
         '7h': 5,
         'Jc': 5,
         'Js': 5,
         '9c': 5,
         '3h': 5,
         '4h': 5,
         '5s': 5,
         '3c': 5,
         'As': 5,
         'Ac': 5,
         '6c': 5,
         '8s': 5,
         '8h': 5,
         'Ah': 5,
         '4c': 5,
         'Jh': 5,
         '2c': 5,
         'Tc': 5,
         '9s': 5,
         '5h': 5,
         '2h': 4,
         '3d': 4,
         'Qs': 4,
         'Ks': 4,
         '2s': 4,
         '5c': 4,
         'Th': 4,
         '4s': 4,
         '6s': 4,
         'Kd': 4,
         '5d': 4,
         '7s': 4,
         '6h': 3,
         '8d': 3,
         '2d': 2})

In [49]:
len(cards) / 52

4.9423076923076925

In [52]:
len(cards) / 52 / 6 * 100

82.37179487179488

In [51]:
# conclusion is that there are 6 decks
# deck penetration is 82.37179487179488